# Vietnamese Extractive News Summarisation

Compare six extractive summarisation methods:

| # | Method | Characteristics |
|---|--------|----------------|
| 1 | **Lead-k** | Baseline: select the first *k* sentences |
| 2 | **Vanilla LexRank** | Graph centrality (TF-IDF cosine similarity) |
| 3 | **Position-Aware LexRank** | LexRank with position and title priors |
| 4 | **Position-Aware LexRank + MMR** | Same as above with MMR for redundancy reduction |
| 5 | **BERT Centroid** | Pure BERT: rank sentences by similarity to the centroid embedding |
| 6 | **PACSUM** | Directed-graph centrality with BERT embeddings (ACL 2019) |

**Evaluation metrics:** ROUGE-1/2/L, Redundancy, Source Coverage, BERTScore.

> **Embedding model:** `paraphrase-multilingual-MiniLM-L12-v2`  
> **Speed:** Embeddings are computed once and cached — all experiments reuse the cache.

## 0. Library Setup (Run Once)

In [ ]:
# !pip install sentence-transformers bert-score tqdm --quiet
# GPU PyTorch (chọn đúng CUDA version):
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126 --quiet

## 1. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  CUDA không khả dụng — đang chạy trên CPU (chậm hơn ~20x)')
    print('   Fix: pip install torch --index-url https://download.pytorch.org/whl/cu126')

✅ GPU: NVIDIA GeForce RTX 4060 Laptop GPU
   VRAM: 8.6 GB


## 2. Import libraries and load data

In [ ]:
from pathlib import Path
import importlib, warnings
from IPython.display import Markdown, display
warnings.filterwarnings('ignore')

import position_aware_lexrank_mmr as sx
sx = importlib.reload(sx)
sx.configure_stdout()

import advanced_summarizer as adv
adv = importlib.reload(adv)

DATA_DIR   = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

groups = sx.collect_article_groups(DATA_DIR)
print(f'Số cluster: {len(groups)}')
print('10 cluster đầu:',
      [(g.group_id, len(g.articles), len(g.references)) for g in groups[:10]])

Số cluster: 300
10 cluster đầu: [('Cluster_001', 10, 2), ('Cluster_002', 5, 2), ('Cluster_003', 10, 2), ('Cluster_004', 6, 2), ('Cluster_005', 5, 2), ('Cluster_006', 10, 2), ('Cluster_007', 10, 2), ('Cluster_008', 5, 2), ('Cluster_009', 5, 2), ('Cluster_010', 6, 2)]


## 3. Initialise Sentence Embedder

The model will be downloaded automatically the first time (~120 MB).

In [ ]:
embedder = adv.SentenceEmbedder('paraphrase-multilingual-MiniLM-L12-v2')
_ = embedder.encode(['Khởi tạo mô hình.'])  # force-load now
print('Embedder sẵn sàng.')

[SentenceEmbedder] Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU
[SentenceEmbedder] Loading 'paraphrase-multilingual-MiniLM-L12-v2' ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6296.99it/s]


[SentenceEmbedder] Model ready.
Embedder sẵn sàng.


## 4. Build Embedding Cache

BERT encoding is performed only once for each cluster.  
All experiments and grid searches reuse the cached embeddings.

Estimated runtime: **~2–3 minutes** on RTX 4060 Laptop GPU.

In [ ]:
import time
t0 = time.time()
emb_cache = adv.build_embeddings_cache(groups, embedder)
elapsed = time.time() - t0
total_sents = sum(len(c.records) for c in emb_cache.values())
print(f'Cache built in {elapsed:.1f}s  |  '
      f'{len(emb_cache)} clusters  |  {total_sents:,} sentences')

Building embedding cache: 100%|██████████| 300/300 [00:21<00:00, 14.14cluster/s]

Cache built in 21.2s  |  300 clusters  |  24,912 sentences


## 5. Utility Functions

In [ ]:
def markdown_table(rows, columns):
    header = '| ' + ' | '.join(title for _, title in columns) + ' |'
    sep    = '| ' + ' | '.join('---' for _ in columns) + ' |'
    body   = []
    for row in rows:
        values = []
        for key, _ in columns:
            v = row.get(key, '')
            values.append(f'{v:.4f}' if isinstance(v, float) else str(v))
        body.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join([header, sep, *body])

## 6. Main Ablation Table

Compare all six methods using default parameters, averaged across all clusters.

TF-IDF methods: **~5 seconds**.  
BERT-based methods: **~15 seconds** (using cache).

In [ ]:
# ── TF-IDF baselines ─────────────────────────────────────────────────────────
baseline_rows = sx.run_experiments(
    groups,
    methods=sx.METHODS,
    max_sentences=5,
    threshold=0.1,
    position_weight=0.8,
    lambda_mmr=0.7,
)
baseline_avg = sx.average_experiment_rows(baseline_rows)

# ── BERT methods (bert_centroid + pacsum) ─────────────────────────────────────
bert_rows = adv.run_experiments_bert(
    groups,
    embedder,
    methods=adv.METHODS_BERT,
    max_sentences=5,
    pacsum_beta=0.0,
    embeddings_cache=emb_cache,
)
bert_avg = adv.average_experiment_rows_bert(bert_rows)

# ── Save CSVs ─────────────────────────────────────────────────────────────────
all_rows = baseline_rows + bert_rows
sx.write_rows_csv(OUTPUT_DIR / 'results.csv',         all_rows)
sx.write_rows_csv(OUTPUT_DIR / 'results_average.csv', baseline_avg + bert_avg)
print(f'Saved {len(all_rows)} rows → outputs/results.csv')

# ── Display ───────────────────────────────────────────────────────────────────
ALL_ORDER = list(sx.METHODS) + list(adv.METHODS_BERT)
ALL_LABELS = {**sx.METHOD_LABELS, **adv.METHOD_LABELS_BERT}
combined_avg = sorted(
    baseline_avg + bert_avg,
    key=lambda r: ALL_ORDER.index(str(r['method'])) if str(r['method']) in ALL_ORDER else 99
)

display(Markdown(markdown_table(combined_avg, [
    ('method_label',   'Method'),
    ('rouge1_f1',      'ROUGE-1 F1'),
    ('rouge2_f1',      'ROUGE-2 F1'),
    ('rougel_f1',      'ROUGE-L F1'),
    ('redundancy',     'Redundancy'),
    ('source_coverage','SrcCover'),
])))

Saved 1800 rows → outputs/results.csv


| Method | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 | Redundancy | SrcCover |
| --- | --- | --- | --- | --- | --- |
| Lead-k | 0.4513 | 0.2731 | 0.2897 | 0.0945 | 0.1756 |
| Vanilla LexRank | 0.4656 | 0.2840 | 0.2862 | 0.3307 | 0.6022 |
| Position-Aware LexRank | 0.4690 | 0.2977 | 0.2922 | 0.3369 | 0.6416 |
| Position-Aware LexRank + MMR | 0.5135 | 0.3354 | 0.3044 | 0.2147 | 0.6166 |
| BERT Centroid | 0.4474 | 0.2609 | 0.2656 | 0.6933 | 0.5972 |
| PACSUM | 0.4524 | 0.2637 | 0.2705 | 0.5176 | 0.2578 |

## 7. Detailed Results by Cluster (Preview)

Display the first 5 clusters only. Full results are saved in `outputs/results.csv`.

In [ ]:
DETAIL_CLUSTERS = 5
preview_ids = {g.group_id for g in groups[:DETAIL_CLUSTERS]}
detail_preview = [
    r for r in sorted(all_rows, key=lambda r: (r['file'], r['method']))
    if r['file'] in preview_ids
]
print(f'Hiển thị {len(detail_preview)} dòng '
      f'({DETAIL_CLUSTERS}/{len(groups)} cluster). Full: outputs/results.csv')
display(Markdown(markdown_table(detail_preview, [
    ('file',           'Cluster'),
    ('method_label',   'Method'),
    ('rouge1_f1',      'R1 F1'),
    ('rouge2_f1',      'R2 F1'),
    ('rougel_f1',      'RL F1'),
    ('redundancy',     'Redund.'),
    ('source_coverage','SrcCover'),
])))

Hiển thị 30 dòng (5/300 cluster). Full: outputs/results.csv


| Cluster | Method | R1 F1 | R2 F1 | RL F1 | Redund. | SrcCover |
| --- | --- | --- | --- | --- | --- | --- |
| Cluster_001 | BERT Centroid | 0.3631 | 0.1562 | 0.1661 | 0.6684 | 0.3000 |
| Cluster_001 | Lead-k | 0.2786 | 0.1150 | 0.1473 | 0.1400 | 0.1000 |
| Cluster_001 | Vanilla LexRank | 0.4285 | 0.1543 | 0.1515 | 0.1114 | 0.5000 |
| Cluster_001 | PACSUM | 0.4427 | 0.1985 | 0.2442 | 0.5912 | 0.2000 |
| Cluster_001 | Position-Aware LexRank | 0.4616 | 0.1750 | 0.1578 | 0.1047 | 0.5000 |
| Cluster_001 | Position-Aware LexRank + MMR | 0.4616 | 0.1750 | 0.1578 | 0.1047 | 0.5000 |
| Cluster_002 | BERT Centroid | 0.5059 | 0.3199 | 0.3208 | 0.6208 | 0.4000 |
| Cluster_002 | Lead-k | 0.4323 | 0.2648 | 0.2663 | 0.1057 | 0.2000 |
| Cluster_002 | Vanilla LexRank | 0.4989 | 0.2966 | 0.2836 | 0.2144 | 0.6000 |
| Cluster_002 | PACSUM | 0.3593 | 0.1440 | 0.1690 | 0.4982 | 0.4000 |
| Cluster_002 | Position-Aware LexRank | 0.4989 | 0.2966 | 0.2836 | 0.2144 | 0.6000 |
| Cluster_002 | Position-Aware LexRank + MMR | 0.4989 | 0.2966 | 0.2836 | 0.2144 | 0.6000 |
| Cluster_003 | BERT Centroid | 0.5122 | 0.3084 | 0.2744 | 0.7382 | 0.5000 |
| Cluster_003 | Lead-k | 0.5312 | 0.3425 | 0.3628 | 0.0248 | 0.1000 |
| Cluster_003 | Vanilla LexRank | 0.5221 | 0.3526 | 0.2785 | 0.1971 | 0.3000 |
| Cluster_003 | PACSUM | 0.4483 | 0.1812 | 0.1971 | 0.5872 | 0.1000 |
| Cluster_003 | Position-Aware LexRank | 0.4340 | 0.2291 | 0.2216 | 0.2079 | 0.4000 |
| Cluster_003 | Position-Aware LexRank + MMR | 0.4412 | 0.2347 | 0.2793 | 0.1869 | 0.4000 |
| Cluster_004 | BERT Centroid | 0.3694 | 0.1418 | 0.1829 | 0.7262 | 0.3333 |
| Cluster_004 | Lead-k | 0.4022 | 0.2309 | 0.1981 | 0.0522 | 0.1667 |
| Cluster_004 | Vanilla LexRank | 0.3428 | 0.1240 | 0.1902 | 0.3073 | 0.3333 |
| Cluster_004 | PACSUM | 0.3557 | 0.0904 | 0.1311 | 0.4770 | 0.1667 |
| Cluster_004 | Position-Aware LexRank | 0.3724 | 0.1437 | 0.1802 | 0.1372 | 0.5000 |
| Cluster_004 | Position-Aware LexRank + MMR | 0.3763 | 0.1602 | 0.1935 | 0.0901 | 0.5000 |
| Cluster_005 | BERT Centroid | 0.5658 | 0.4059 | 0.4350 | 0.7515 | 0.8000 |
| Cluster_005 | Lead-k | 0.5328 | 0.3311 | 0.3856 | 0.0841 | 0.2000 |
| Cluster_005 | Vanilla LexRank | 0.5231 | 0.3922 | 0.4242 | 0.5585 | 0.8000 |
| Cluster_005 | PACSUM | 0.5717 | 0.4337 | 0.3607 | 0.5405 | 0.2000 |
| Cluster_005 | Position-Aware LexRank | 0.5089 | 0.3605 | 0.3908 | 0.3700 | 0.8000 |
| Cluster_005 | Position-Aware LexRank + MMR | 0.6332 | 0.5288 | 0.3865 | 0.1866 | 1.0000 |

## 8. Grid Search — Position-Aware LexRank + MMR (TF-IDF)

Analyse the effects of `lambda_mmr`, `threshold`, and `position_weight`.

In [ ]:
tfidf_grid = []
for lambda_mmr in [0.5, 0.6, 0.7, 0.8, 0.9]:
    for threshold in [0.05, 0.10, 0.15, 0.20]:
        for position_weight in [0.6, 0.7, 0.8, 0.9]:
            trial = sx.run_experiments(
                groups,
                methods=('position_lexrank_mmr',),
                max_sentences=5,
                threshold=threshold,
                position_weight=position_weight,
                lambda_mmr=lambda_mmr,
            )
            avg = sx.average_experiment_rows(trial)
            if avg:
                tfidf_grid.append({
                    'lambda_mmr':      lambda_mmr,
                    'threshold':       threshold,
                    'position_weight': position_weight,
                    'rouge1_f1':       avg[0]['rouge1_f1'],
                    'rouge2_f1':       avg[0]['rouge2_f1'],
                    'rougel_f1':       avg[0]['rougel_f1'],
                    'redundancy':      avg[0]['redundancy'],
                    'source_coverage': avg[0]['source_coverage'],
                })

tfidf_grid = sorted(tfidf_grid, key=lambda r: (-r['rougel_f1'], r['redundancy']))
print('Top-10 cấu hình Position-Aware LexRank + MMR:')
display(Markdown(markdown_table(tfidf_grid[:10], [
    ('lambda_mmr',      'lambda'),
    ('threshold',       'threshold'),
    ('position_weight', 'pos_weight'),
    ('rouge1_f1',       'ROUGE-1 F1'),
    ('rouge2_f1',       'ROUGE-2 F1'),
    ('rougel_f1',       'ROUGE-L F1'),
    ('redundancy',      'Redundancy'),
    ('source_coverage', 'SrcCover'),
])))

Top-10 cấu hình Position-Aware LexRank + MMR:


| lambda | threshold | pos_weight | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 | Redundancy | SrcCover |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0.6000 | 0.0500 | 0.7000 | 0.5300 | 0.3508 | 0.3140 | 0.1782 | 0.6061 |
| 0.6000 | 0.0500 | 0.6000 | 0.5286 | 0.3478 | 0.3126 | 0.1822 | 0.6008 |
| 0.6000 | 0.0500 | 0.9000 | 0.5272 | 0.3475 | 0.3119 | 0.1703 | 0.6237 |
| 0.6000 | 0.0500 | 0.8000 | 0.5271 | 0.3470 | 0.3117 | 0.1755 | 0.6143 |
| 0.5000 | 0.0500 | 0.8000 | 0.5304 | 0.3465 | 0.3108 | 0.1479 | 0.6094 |
| 0.5000 | 0.0500 | 0.9000 | 0.5295 | 0.3466 | 0.3103 | 0.1444 | 0.6166 |
| 0.7000 | 0.0500 | 0.7000 | 0.5195 | 0.3408 | 0.3101 | 0.2074 | 0.6139 |
| 0.5000 | 0.1000 | 0.8000 | 0.5288 | 0.3448 | 0.3100 | 0.1557 | 0.6030 |
| 0.5000 | 0.2000 | 0.7000 | 0.5199 | 0.3446 | 0.3099 | 0.1434 | 0.5902 |
| 0.5000 | 0.1000 | 0.6000 | 0.5275 | 0.3435 | 0.3097 | 0.1623 | 0.5895 |

## 9. Grid Search — PACSUM

Analyse the effect of `pacsum_beta`  
(0 = maximum position bias, 1 = symmetric).

Estimated runtime: **< 1 minute** (cache reused, no re-encoding).

In [ ]:
from tqdm.auto import tqdm

pacsum_grid = []
for pacsum_beta in tqdm([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                        desc='PACSUM beta'):
    trial = adv.run_experiments_bert(
        groups, embedder,
        methods=('pacsum',),
        max_sentences=5,
        pacsum_beta=pacsum_beta,
        embeddings_cache=emb_cache,
    )
    avg = adv.average_experiment_rows_bert(trial, methods=('pacsum',))
    if avg:
        pacsum_grid.append({
            'pacsum_beta':    pacsum_beta,
            'rouge1_f1':      avg[0]['rouge1_f1'],
            'rouge2_f1':      avg[0]['rouge2_f1'],
            'rougel_f1':      avg[0]['rougel_f1'],
            'redundancy':     avg[0]['redundancy'],
            'source_coverage':avg[0]['source_coverage'],
        })

pacsum_grid = sorted(pacsum_grid, key=lambda r: -r['rougel_f1'])
display(Markdown(markdown_table(pacsum_grid, [
    ('pacsum_beta',    'beta'),
    ('rouge1_f1',      'ROUGE-1 F1'),
    ('rouge2_f1',      'ROUGE-2 F1'),
    ('rougel_f1',      'ROUGE-L F1'),
    ('redundancy',     'Redundancy'),
    ('source_coverage','SrcCover'),
])))

PACSUM beta: 100%|██████████| 11/11 [00:35<00:00,  3.22s/it]


| beta | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 | Redundancy | SrcCover |
| --- | --- | --- | --- | --- | --- |
| 0.4000 | 0.4580 | 0.2713 | 0.2745 | 0.5583 | 0.3081 |
| 0.5000 | 0.4582 | 0.2714 | 0.2730 | 0.5736 | 0.3361 |
| 0.6000 | 0.4596 | 0.2734 | 0.2727 | 0.5880 | 0.3653 |
| 0.2000 | 0.4563 | 0.2682 | 0.2725 | 0.5341 | 0.2778 |
| 0.1000 | 0.4557 | 0.2675 | 0.2723 | 0.5252 | 0.2707 |
| 0.3000 | 0.4555 | 0.2672 | 0.2711 | 0.5454 | 0.2923 |
| 0.7000 | 0.4586 | 0.2713 | 0.2709 | 0.6084 | 0.4068 |
| 0.0000 | 0.4524 | 0.2637 | 0.2705 | 0.5176 | 0.2578 |
| 0.8000 | 0.4575 | 0.2699 | 0.2701 | 0.6367 | 0.4689 |
| 0.9000 | 0.4530 | 0.2648 | 0.2681 | 0.6722 | 0.5459 |
| 1.0000 | 0.4476 | 0.2611 | 0.2656 | 0.6937 | 0.5999 |


## 10. BERTScore — Semantic Evaluation

BERTScore measures deeper semantic similarity using contextual BERT embeddings.

Estimated runtime: **~4–5 minutes**  
(forward pass of `bert-base-multilingual-cased` on 300 × 6 pairs).

In [ ]:
bertscore_specs = [
    ('Lead-k',                          False, 'lead'),
    ('Vanilla LexRank',                 False, 'lexrank'),
    ('Position-Aware LexRank',          False, 'position_lexrank'),
    ('Position-Aware LexRank + MMR',    False, 'position_lexrank_mmr'),
    ('BERT Centroid',                   True,  'bert_centroid'),
    ('PACSUM',                          True,  'pacsum'),
]

bs_results = []
for label, is_bert, method_key in bertscore_specs:
    candidates, references = [], []
    for group in groups:
        if not group.references:
            continue
        if is_bert:
            result = adv.summarize_group_bert(
                group, embedder, method=method_key, max_sentences=5,
                pacsum_beta=0.0,
                cluster_cache=emb_cache.get(group.group_id),
            )
        else:
            result = sx.summarize_group(
                group, method=method_key, max_sentences=5,
                threshold=0.1, position_weight=0.8, lambda_mmr=0.7,
            )
        if result.text:
            candidates.append(result.text)
            references.append(group.reference_summary)

    bs = adv.evaluate_with_bertscore(candidates, references, lang='vi')
    bs_results.append({'method_label': label, **bs})
    print(f'  {label}: F1={bs["bertscore_f1"]:.4f}')

display(Markdown(markdown_table(bs_results, [
    ('method_label',  'Method'),
    ('bertscore_p',   'BERTScore P'),
    ('bertscore_r',   'BERTScore R'),
    ('bertscore_f1',  'BERTScore F1'),
])))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4168.75it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Lead-k: F1=0.7363


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8919.57it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Vanilla LexRank: F1=0.7366


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11320.27it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10129.94it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  BERT Centroid: F1=0.7314


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10197.14it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  PACSUM: F1=0.7299


| Method | BERTScore P | BERTScore R | BERTScore F1 |
| --- | --- | --- | --- |
| Lead-k | 0.7697 | 0.7061 | 0.7363 |
| Vanilla LexRank | 0.7767 | 0.7011 | 0.7366 |
| Position-Aware LexRank | 0.7859 | 0.7092 | 0.7452 |
| Position-Aware LexRank + MMR | 0.7907 | 0.7265 | 0.7569 |
| BERT Centroid | 0.7663 | 0.7002 | 0.7314 |
| PACSUM | 0.7608 | 0.7019 | 0.7299 |

## 11. View Sample Summaries — All 6 Methods on Cluster_001

Display sample summaries generated by all six methods on Cluster_001.

Estimated runtime: **< 5 seconds** (using cache).

In [ ]:
sample_group = groups[0]
sample_cache = emb_cache.get(sample_group.group_id)

print(f'Cluster : {sample_group.group_id}')
print(f'Articles: {len(sample_group.articles)}  |  Refs: {len(sample_group.references)}')
print('\n── Reference summary ──')
print(sample_group.reference_summary)
print()

# TF-IDF baselines
for method in sx.METHODS:
    result  = sx.summarize_group(
        sample_group, method=method, max_sentences=5,
        threshold=0.1, position_weight=0.8, lambda_mmr=0.7,
    )
    scores  = sx.evaluate_against_references(result.text, sample_group.references)
    src_cov = len(set(result.selected_sources or [])) / len(sample_group.articles)
    print('=' * 90)
    print(f'[TF-IDF] {sx.METHOD_LABELS[method]}')
    print(f'ROUGE-L: {scores["rougel_f1"]:.4f}  |  '
          f'Redundancy: {result.redundancy:.4f}  |  SrcCover: {src_cov:.4f}')
    print(result.text)
    print()

# BERT methods
for method in adv.METHODS_BERT:
    result  = adv.summarize_group_bert(
        sample_group, embedder, method=method, max_sentences=5,
        pacsum_beta=0.0, cluster_cache=sample_cache,
    )
    scores  = sx.evaluate_against_references(result.text, sample_group.references)
    src_cov = len(set(result.selected_sources or [])) / len(sample_group.articles)
    print('=' * 90)
    print(f'[BERT]   {adv.METHOD_LABELS_BERT[method]}')
    print(f'ROUGE-L: {scores["rougel_f1"]:.4f}  |  '
          f'Redundancy: {result.redundancy:.4f}  |  SrcCover: {src_cov:.4f}')
    print(result.text)
    print()

Cluster : Cluster_001
Articles: 10  |  Refs: 2

── Reference summary ──
Chiếc Airbus A320 (chuyến bay số hiệu MS804) bay từ thủ đô Paris đến thủ đô Cairo thì biến mất khỏi màn hình radar vào sáng sớm ngày 19.5.

Các chuyên gia hàng không lo ngại một hành động cố ý đã khiến máy bay biến mất khỏi màn hình radar.
Bộ trưởng Hàng không Ai Cập Sherif Fathy nghiêng về khả năng MS804 bị khủng bố nhiều hơn là trục trặc kỹ thuật.
Họ nghi ngờ một quả bom khiến máy bay nổ tung, sau đó rơi xuống biển.

Tổng thư ký Tổ chức Hiệp ước Bắc Đại Tây Dương Jens Stoltenberg cho biết sẽ hỗ trợ công tác tìm kiếm máy bay MS 804.

một cuộc điều tra quy mô lớn đang được tiến hành sau khi chiếc A320 biến mất.
Theo các nguồn tin, hải mảnh nhựa vỡ được tìm thấy có màu trắng và đỏ.
Những mảnh vỡ  được phát hiện ở gần khu vực nơi hệ thống định vị khẩn cấp trên máy bay đã phát tín hiệu trước đó.

66 người được cho là đã chết khi chuyến bay MS804 biến mất.

Ezeddin Samar là nữ tiếp viên có mặt trên chuyến bay MS804.
Ng